In [1]:
# %%
"""
📌 Notebook: 02_data_preprocessing.ipynb

🎯 Purpose:
Clean the merged dataset and prepare high-quality sensor signals
for feature engineering and sequence modeling.

Key Steps:
1. Remove irrelevant columns
2. Handle missing values
3. Sort time-series data
4. Remove static/redundant sensor readings
5. Handle outliers (VERY IMPORTANT)
6. Validate data distribution
"""

import pandas as pd
import numpy as np
import os

# Load dataset
input_path = "../processed_data/merged_dataset.csv"
df = pd.read_csv(input_path, low_memory=False)

print(f"✅ Dataset loaded. Initial shape: {df.shape}")

✅ Dataset loaded. Initial shape: (2987342, 11)


In [2]:
# %%
"""
🧹 Step 1: Select relevant columns

We only keep sensor readings + labels.
Other columns like ID, Timestamp are not needed for modeling.
"""
columns_to_keep = [
    'X_Acc', 'Y_Acc', 'Z_Acc', 
    'X_Gyro', 'Y_Gyro', 'Z_Gyro', 
    'rating', 'trip_id'
]

df_cleaned = df[columns_to_keep].copy()

print("✅ Irrelevant columns removed.")
print("Remaining columns:", df_cleaned.columns.tolist())

✅ Irrelevant columns removed.
Remaining columns: ['X_Acc', 'Y_Acc', 'Z_Acc', 'X_Gyro', 'Y_Gyro', 'Z_Gyro', 'rating', 'trip_id']


In [3]:
# %%
"""
⚠️ Step 2: Handle missing values

Missing values can break neural networks.
We remove rows with NaNs.
"""

null_counts = df_cleaned.isnull().sum()
print("🔍 Null values per column:\n", null_counts)

# Drop NaNs
df_cleaned = df_cleaned.dropna()

print(f"✅ Shape after dropping nulls: {df_cleaned.shape}")

🔍 Null values per column:
 X_Acc      0
Y_Acc      0
Z_Acc      0
X_Gyro     0
Y_Gyro     0
Z_Gyro     0
rating     0
trip_id    0
dtype: int64
✅ Shape after dropping nulls: (2987342, 8)


In [4]:
# %%
"""
📊 Step 3: Sort data (CRITICAL for time-series)
We must ensure data is in correct time order within each trip.

"""

if 'Timestamp' in df.columns:
    df_cleaned['Timestamp'] = df['Timestamp']
    df_cleaned = df_cleaned.sort_values(by=['trip_id', 'Timestamp']).reset_index(drop=True)
else:
    df_cleaned = df_cleaned.sort_values(by=['trip_id']).reset_index(drop=True)

print("✅ Data sorted by trip_id and time")

✅ Data sorted by trip_id and time


In [5]:
print("Before static removal:", df_cleaned.shape)

Before static removal: (2987342, 9)


In [6]:
# %%
"""
🚫 Step 4: Remove static sensor noise (Optimized Version)

🎯 Problem:
Sensor readings sometimes remain exactly the same across consecutive rows.
This usually happens when:
- Vehicle is idle
- Sensor is not updating properly

These repeated values add NO useful information and can harm model learning.

🚀 Solution:
We remove rows where ALL sensor values are identical to the previous row
within the same trip.

⚡ Optimization:
- Uses vectorized operations (faster than groupby-apply)
- Avoids pandas FutureWarning
- Scales well for large datasets (millions of rows)

Handles floating-point precision issues using tolerance comparison.

"""

sensor_cols = ['X_Acc', 'Y_Acc', 'Z_Acc', 'X_Gyro', 'Y_Gyro', 'Z_Gyro']

initial_rows = df_cleaned.shape[0]

# Shift values within each trip
shifted = df_cleaned.groupby('trip_id')[sensor_cols].shift()

# ✅ Use tolerance comparison instead of ==
tolerance = 1e-4  # you can tune this

diff = (df_cleaned[sensor_cols] - shifted).abs()

# Row is static if ALL differences are very small
static_mask = (diff < tolerance).all(axis=1)

# Remove static rows
df_cleaned = df_cleaned[~static_mask]

final_rows = df_cleaned.shape[0]

print(f"✅ Shape after removing static data: {df_cleaned.shape}")
print(f"🧹 Removed {initial_rows - final_rows} redundant rows")

✅ Shape after removing static data: (2750848, 9)
🧹 Removed 236494 redundant rows


In [7]:
print("After static removal:", df_cleaned.shape)

After static removal: (2750848, 9)


In [8]:
# %%
"""
🚨 Step 5: Outlier Handling (VERY IMPORTANT)

Extreme values (e.g., gyro spikes) can destabilize neural networks.

We use percentile-based clipping for robustness.
"""

# Calculate percentile limits
lower = df_cleaned[sensor_cols].quantile(0.01)
upper = df_cleaned[sensor_cols].quantile(0.99)

# Apply clipping
df_cleaned[sensor_cols] = df_cleaned[sensor_cols].clip(lower, upper, axis=1)

print("✅ Outliers handled using percentile clipping (1% - 99%)")

✅ Outliers handled using percentile clipping (1% - 99%)


In [9]:
# %%
"""
📈 Step 6: Data validation
"""

print("📊 Statistical summary:")
display(df_cleaned.describe())

print("\n📊 Samples per rating:")
print(df_cleaned['rating'].value_counts().sort_index())

📊 Statistical summary:


,X_Acc,Y_Acc,Z_Acc,X_Gyro,Y_Gyro,Z_Gyro,rating,trip_id,Timestamp
count,2.750848e+06,2.750848e+06,2.750848e+06,2.750848e+06,2.750848e+06,2.750848e+06,2.750848e+06,2.750848e+06,2.750848e+06
mean,4.467363e-01,2.474485e+00,7.950558e+00,9.983163e-01,-4.122188e-01,6.674443e-01,3.037350e+00,9.638691e+01,1.770406e+12
std,2.967555e+00,3.625341e+00,3.396673e+00,2.530141e+01,2.349687e+01,2.098205e+01,1.324187e+00,5.713190e+01,6.129591e+09
min,-8.605078e+00,-6.577851e+00,-4.723760e+00,-1.072435e+02,-1.079951e+02,-9.327500e+01,1.000000e+00,1.000000e+00,1.739352e+12
25%,-7.864614e-01,-1.028871e-02,6.886109e+00,-4.088770e+00,-5.152320e+00,-3.490030e+00,2.000000e+00,4.600000e+01,1.771405e+12
50%,7.296608e-02,1.550255e+00,9.055335e+00,4.338905e-02,7.576567e-03,2.799000e-02,3.000000e+00,9.200000e+01,1.771521e+12
75%,1.160973e+00,5.113950e+00,9.882222e+00,5.084904e+00,5.035400e+00,4.318083e+00,4.000000e+00,1.450000e+02,1.771527e+12
max,1.130348e+01,1.282091e+01,1.493400e+01,1.181269e+02,1.001560e+02,1.023179e+02,5.000000e+00,1.920000e+02,1.774003e+12



📊 Samples per rating:
rating
1    496627
2    457675
3    657628
4    724162
5    414756
Name: count, dtype: int64


In [10]:
# %%
"""
💾 Step 7: Save cleaned dataset
"""

output_path = "../processed_data/cleaned_dataset.csv"
df_cleaned.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved at: {output_path}")

✅ Cleaned dataset saved at: ../processed_data/cleaned_dataset.csv
